# EDA — All CVE Records (chat)

**Dataset:** [`AlicanKiraz0/All-CVE-Records-Training-Dataset`](https://huggingface.co/datasets/AlicanKiraz0/All-CVE-Records-Training-Dataset)

≈300 000 registros CVE (1999–2025) convertidos a conversaciones system/user/assistant para fine-tuning.

**Formato:** JSON Lines (`all_cve_database.jsonl`). Columnas: `System`, `User`, `Assistant`.

**Preguntas guía para el EDA**
- ¿Cuántos registros hay por año de CVE?
- ¿Qué CWE son más frecuentes y cómo se distribuye el CVSS?
- ¿Las respuestas siguen una plantilla fija? ¿Qué campos incluyen?
- ¿Cuántos CVE tienen varios turnos?

In [ ]:
import sys, json, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Busca la raíz del repo (donde está src/profiler.py) y la agrega al path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "profiler.py").exists())
sys.path.insert(0, str(ROOT))
from src.profiler import *

sns.set_theme(style="whitegrid", palette="tab10")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

ctx = eda_context("04")   # rutas: datos en data/raw/..., resultados en esta carpeta
print(ctx)

## 1. Inventario de archivos

In [ ]:
files = list_files(ctx.data_dir)
display(summarize_extensions(files))
display(files)

## 2. Cargar (muestra)
El archivo es grande, así que por defecto se toma una **muestra aleatoria** de `SAMPLE_N` filas
(se recorre el archivo completo pero solo se guarda la muestra en memoria).
Cambia `FULL = True` para cargarlo entero si tu equipo tiene RAM suficiente.

In [ ]:
PATH = ctx.data_dir / "all_cve_database.jsonl"
SAMPLE_N = 20_000
FULL = False

print("Primer registro:"); display(peek_record(PATH))
N_TOTAL = count_lines(PATH)
print(f"Filas totales: {N_TOTAL:,}")

df = read_jsonl(PATH) if FULL else read_jsonl(PATH, sample=SAMPLE_N)
print("Filas cargadas:", len(df))
display(df.head(3))
display(profile_df(df))

## 3. Largo de los textos

In [ ]:
L = text_lengths(df, ["System", "User", "Assistant"])
display(L.describe().round(1))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, c in zip(axes, L.columns):
    sns.histplot(L[c].clip(lower=1), bins=50, log_scale=True, ax=ax); ax.set_title(f"{c} (caracteres, log)")
plt.tight_layout(); save_fig(fig, ctx, "largo_textos"); plt.show()

## 4. Prompts de sistema, duplicados y plantillas

In [ ]:
print("System prompts distintos:", df["System"].nunique())
display(df["System"].str[:150].value_counts().head(10).to_frame("n"))

display(duplicates_report(df, {"User": ["User"], "Assistant": ["Assistant"], "User + Assistant": ["User", "Assistant"]}))

# Señal de plantillas: primeras 4 palabras más comunes del mensaje del usuario
first_words = df["User"].fillna("").str.split().str[:4].str.join(" ")
display(first_words.value_counts().head(15).to_frame("n"))

In [ ]:
# ¿Cuántas respuestas son negativas/rechazos? (patrón aproximado)
REFUSAL = r"(?i)\b(?:I can(?:no|')t|I won't|I'm sorry|I am sorry|I must decline|I can't assist)\b"
ref = df["Assistant"].fillna("").str.contains(REFUSAL)
print(f"Respuestas con patrón de rechazo: {ref.sum():,} ({ref.mean():.1%})")

## 5. Información específica de CVE
Se extraen del texto el ID del CVE, el año, los CWE y los puntajes CVSS con expresiones regulares (aproximado: revisa los ejemplos).

In [ ]:
# Ojo: el README usa guiones no estándar (‑ U+2011), por eso la clase [-‑‐]
CVE_RE = r"CVE[-‑‐](\d{4})[-‑‐](\d{4,})"
ids = df["User"].str.extract(CVE_RE)
ids_a = df["Assistant"].str.extract(CVE_RE)
ids = ids.fillna(ids_a)
df["cve_year"] = pd.to_numeric(ids[0], errors="coerce")
df["cve_id"] = "CVE-" + ids[0] + "-" + ids[1]
print(f"Filas con CVE identificado: {df['cve_id'].notna().mean():.1%}")
print("CVE distintos en la muestra:", df["cve_id"].nunique())

fig, ax = plt.subplots(figsize=(11, 4))
df["cve_year"].value_counts().sort_index().plot.bar(ax=ax); ax.set_title("Registros por año del CVE (muestra)")
save_fig(fig, ctx, "cve_por_anio"); plt.show()

In [ ]:
# Campos de metadatos que aparecen en las respuestas (líneas tipo "- **Campo**: valor")
campos = df["Assistant"].str.findall(r"(?m)^\s*[-*]\s*\*\*(.+?)\*\*\s*:").explode()
display(campos.value_counts().head(25).to_frame("n_filas"))

In [ ]:
cwe = df["Assistant"].str.findall(r"CWE-\d+").explode()
display(cwe.value_counts().head(20).to_frame("n"))

cvss = pd.to_numeric(df["Assistant"].str.extract(r"(?i)(?:base\s*score|cvss[^\n]{0,40}?)[:\s]*(\d{1,2}\.\d)")[0], errors="coerce")
cvss = cvss[(cvss >= 0) & (cvss <= 10)]
print(f"Filas con algún puntaje CVSS detectado: {len(cvss):,}")
if len(cvss):
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(cvss, bins=20, ax=ax); ax.set_title("Puntaje CVSS detectado (aprox.)")
    save_fig(fig, ctx, "cvss"); plt.show()

In [ ]:
# Multi-turno: varias líneas comparten el mismo CVE
por_cve = df.groupby("cve_id").size()
display(por_cve.describe())
df = df.drop(columns=["cve_year", "cve_id"])

## Ejemplos

In [ ]:
show_examples(df, n=2, cols=["User", "Assistant"], max_chars=800)

## Guardar resultados
Se guardan en `tables/` de esta carpeta. `resumen.csv` lo consolida `eda/00_general`.

In [ ]:
save_table(profile_df(df), ctx, "perfil_columnas")
display(save_summary(ctx, "AlicanKiraz0/All-CVE-Records-Training-Dataset", df, total_rows=N_TOTAL, dup_cols=["User", "Assistant"],
                     notas="análisis sobre muestra aleatoria"))

## Próximos pasos / notas
- Anota tus hallazgos en el `README.md` de esta carpeta.
-